This notebook builds on IAM fundamentals and covers the advanced mechanisms that Solutions Architects use to design secure, multi-account AWS environments: permission boundaries, resource-based policies, AWS STS, cross-account access patterns, and AWS Organizations with Service Control Policies.

## Identity-based vs Resource-based Policies

There are two places a policy can live:

| | Identity-based | Resource-based |
|---|---|---|
| **Attached to** | IAM user, group, or role | The AWS resource itself |
| **Controls** | What the identity can do | Who can access this resource |
| **Examples** | Managed policies, inline policies | S3 bucket policy, SQS queue policy, KMS key policy |
| **Cross-account** | Requires a role assumption | Can directly grant access to another account |

### Cross-account access: two approaches

**Approach A — Role assumption (identity-based):**
Account A's user assumes a role in Account B. Account B must trust Account A in the role's trust policy, and Account A must allow the user to call `sts:AssumeRole`.

**Approach B — Resource-based policy:**
Attach a bucket policy that directly allows Account A's principal. No role assumption needed — the principal accesses the resource directly.

> **Key difference:** With role assumption, the principal *becomes* the role and loses their original identity. With a resource-based policy, they keep their own identity.

## IAM Policy Evaluation Logic

When AWS receives an API call, it evaluates all applicable policies in this order:

1. **Explicit Deny** — any policy denies → request is denied immediately
2. **SCPs** (if AWS Organizations is in use) — must allow the action
3. **Resource-based policy** — if it grants access to the same account, allow
4. **IAM Permission Boundary** — must allow the action (if set)
5. **Session policies** (if using STS) — must allow the action
6. **Identity-based policy** — if it allows the action, allow
7. **Implicit Deny** — nothing allowed it → request is denied

The practical rule: the effective permissions are the **intersection** of all applicable policy layers, minus any explicit denies.

## Permission Boundaries

A **permission boundary** sets the maximum permissions an IAM entity (user or role) can ever have — regardless of what identity-based policies are attached to it.

```
Effective permissions = identity-based policy  ∩  permission boundary
```

### When to use them

The classic use case is **delegated administration**: you want to allow a developer to create IAM roles for their service, but prevent them from creating roles that are more powerful than their own access.

```
Admin grants developer:
  - Identity policy: iam:CreateRole, iam:AttachRolePolicy
  - Boundary condition: new roles must have the boundary attached

Developer creates a Lambda execution role:
  - Identity policy on role: AmazonDynamoDBFullAccess
  - Permission boundary on role: only S3 + DynamoDB allowed
  → Effective: DynamoDB only (intersection)
```

> Permission boundaries do **not** grant permissions on their own — they only cap what can be granted.

## IAM Policy Conditions

Conditions add context-aware restrictions to a policy statement. Common condition keys:

| Condition key | Example use |
|---|---|
| `aws:SourceIp` | Allow only from a specific IP range |
| `aws:RequestedRegion` | Restrict actions to specific regions |
| `aws:MultiFactorAuthPresent` | Require MFA for sensitive operations |
| `aws:PrincipalTag` | Allow if the principal has a specific tag |
| `s3:prefix` | Restrict S3 access to a specific key prefix |
| `iam:PassedToService` | Control which services a role can be passed to |

### Example — require MFA for S3 delete

```json
{
  "Effect": "Deny",
  "Action": "s3:DeleteObject",
  "Resource": "*",
  "Condition": {
    "BoolIfExists": { "aws:MultiFactorAuthPresent": "false" }
  }
}
```

This denies S3 delete if MFA was not used during authentication.

## AWS STS — Security Token Service

**STS** is the service that issues temporary security credentials when a role is assumed. Credentials consist of:
- `AccessKeyId`
- `SecretAccessKey`  
- `SessionToken`
- `Expiration` (15 minutes to 12 hours)

### STS API calls

| API | Use case |
|---|---|
| `AssumeRole` | Cross-account or same-account role assumption |
| `AssumeRoleWithWebIdentity` | Federation with web identity providers (Amazon Cognito, Google, Facebook) |
| `AssumeRoleWithSAML` | Federation with corporate identity providers (Active Directory via SAML 2.0) |
| `GetSessionToken` | MFA-enabled access for an IAM user |
| `GetFederationToken` | Issue credentials to a custom-built identity broker |

### Instance profiles
When you attach an IAM role to an EC2 instance, AWS wraps it in an **instance profile**. The EC2 metadata service (at `169.254.169.254`) vends rotating temporary credentials to code running on the instance automatically — no access keys needed.

## AWS Organizations

**AWS Organizations** lets you centrally manage multiple AWS accounts under a single umbrella.

```
Root
├── Management Account (formerly master)
├── OU: Production
│   ├── Account: prod-web
│   └── Account: prod-data
├── OU: Development
│   ├── Account: dev-web
│   └── Account: dev-data
└── OU: Security
    └── Account: security-tooling
```

### Key concepts

| Concept | Description |
|---|---|
| **Management account** | The account that created the organization; pays all bills; cannot be restricted by SCPs |
| **Member account** | Any other account in the organization |
| **Organizational Unit (OU)** | A logical container for accounts; can be nested |
| **Root** | The top-level container — parent of all OUs and accounts |

### Benefits
- **Consolidated billing** — one bill for all accounts; volume discounts apply across the org
- **Centralized policy enforcement** — via SCPs (see below)
- **Account vending** — create new accounts programmatically via the API
- **Shared services** — share VPCs, Reserved Instances, and Savings Plans across accounts

## Service Control Policies (SCPs)

An **SCP** is a policy attached to the root, an OU, or a specific account in AWS Organizations. It defines the **maximum permissions** available to all principals in that scope — including the root user of member accounts.

### What SCPs do and don't do

| | SCPs |
|---|---|
| Do | Set a ceiling on what any IAM identity in the account can do |
| Do | Apply to all users and roles in member accounts |
| Do NOT | Grant any permissions on their own |
| Do NOT | Apply to the management account |
| Do NOT | Affect service-linked roles |

### Deny-list vs allow-list strategy

**Deny-list (default):** Start with `FullAWSAccess` SCP attached at the root, then attach additional SCPs that explicitly deny specific actions.

**Allow-list:** Remove `FullAWSAccess`, then attach SCPs that explicitly allow only specific actions. More restrictive, but harder to manage.

### Example SCP — prevent leaving the organization

```json
{
  "Effect": "Deny",
  "Action": "organizations:LeaveOrganization",
  "Resource": "*"
}
```

### Example SCP — restrict to specific regions

```json
{
  "Effect": "Deny",
  "NotAction": [
    "iam:*", "organizations:*", "support:*"
  ],
  "Resource": "*",
  "Condition": {
    "StringNotEquals": {
      "aws:RequestedRegion": ["eu-west-1", "eu-central-1"]
    }
  }
}
```

This blocks all actions outside of the two allowed regions, while exempting global services like IAM.

## IAM Access Analyzer

**IAM Access Analyzer** continuously monitors resource-based policies to find resources that are accessible from outside your AWS account or organization — unintended public or cross-account exposure.

Resources it analyses:
- S3 buckets
- IAM roles
- KMS keys
- Lambda functions and layers
- SQS queues
- Secrets Manager secrets

When it finds external access it generates a **finding** — you can review, archive (if intentional), or remediate it.

> Access Analyzer also has a **policy validation** feature that checks policies you write for errors, unused permissions, and adherence to best practices before you deploy them.

## Working with Organizations via boto3

In [ ]:
import boto3
import json

org = boto3.client('organizations')
iam = boto3.client('iam')

In [ ]:
# List all accounts in the organization
paginator = org.get_paginator('list_accounts')
for page in paginator.paginate():
    for account in page['Accounts']:
        print(f"{account['Id']}  {account['Name']:<30}  {account['Status']}")

In [ ]:
# List all SCPs in the organization
response = org.list_policies(Filter='SERVICE_CONTROL_POLICY')
for policy in response['Policies']:
    print(f"{policy['Name']:<40}  {policy['Description']}")

In [ ]:
# Assume a role in another account and list its S3 buckets
sts = boto3.client('sts')

assumed = sts.assume_role(
    RoleArn='arn:aws:iam::123456789012:role/ReadOnlyRole',  # replace with real ARN
    RoleSessionName='cross-account-demo'
)

creds = assumed['Credentials']
s3 = boto3.client(
    's3',
    aws_access_key_id=creds['AccessKeyId'],
    aws_secret_access_key=creds['SecretAccessKey'],
    aws_session_token=creds['SessionToken']
)

buckets = s3.list_buckets()['Buckets']
for b in buckets:
    print(b['Name'])

In [ ]:
# Create a permission boundary policy
boundary_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:*", "dynamodb:*", "logs:*"],
            "Resource": "*"
        }
    ]
}

response = iam.create_policy(
    PolicyName='AppServiceBoundary',
    PolicyDocument=json.dumps(boundary_policy),
    Description='Max permissions for application service roles'
)
print(response['Policy']['Arn'])

## Summary

| Concept | Key Takeaway |
|---|---|
| Identity-based policy | Attached to the principal; controls what it can do |
| Resource-based policy | Attached to the resource; controls who can access it; enables direct cross-account access |
| Permission boundary | Caps the maximum permissions of a user or role; does not grant on its own |
| Policy evaluation | Explicit deny wins; effective permissions = intersection of all policy layers |
| IAM Conditions | Add context to policies — IP, region, MFA, tags |
| STS | Issues temporary credentials when a role is assumed; powers cross-account and federation |
| AWS Organizations | Centrally manage multiple accounts; consolidated billing; account hierarchy via OUs |
| SCPs | Set max permissions for entire OUs or accounts; never apply to management account |
| IAM Access Analyzer | Detects unintended external access to resources; validates policies before deployment |